In [1]:
import pandas as pd

# Load identifiers as text to preserve leading zeros
associations = pd.read_csv(
    "data/processed/territorial_associations.csv",
    dtype="string"
)

funding = pd.read_csv(
    "data/processed/state_funding_2020_2024.csv",
    dtype={
        "siren": "string",
        "siret": "string",
        "department_code": "string",
        "budget_program": "string"
    }
)

print("Association records:", len(associations))
print("Missing association purposes:", associations["objet"].isna().sum())

print("\nFunding records:", len(funding))
print("Funding years:", sorted(funding["year"].unique().tolist()))
print("Missing funding SIRENs:", funding["siren"].isna().sum())

Association records: 522
Missing association purposes: 0

Funding records: 30309
Funding years: [2020, 2021, 2022, 2023, 2024]
Missing funding SIRENs: 2


In [2]:
# Prepare text for quality checks only
purposes = (
    associations["objet"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

word_counts = purposes.str.split().str.len()

print("Empty purposes:", purposes.eq("").sum())
print("Purposes with fewer than 10 words:", word_counts.lt(10).sum())
print("Distinct purposes:", purposes.nunique())
print("Duplicate purposes beyond the first:", purposes.duplicated().sum())

display(word_counts.describe().round(1))

Empty purposes: 0
Purposes with fewer than 10 words: 12
Distinct purposes: 522
Duplicate purposes beyond the first: 0


count    522.0
mean      94.4
std       88.5
min        5.0
25%       38.0
50%       70.5
75%      120.8
max      596.0
Name: objet, dtype: float64

In [3]:
# Inspect short purposes before deciding how to use them
display(
    associations.loc[
        word_counts.lt(10),
        ["titre", "objet"]
    ]
)

,titre,objet
44,IMPULSION,promouvoir et accompagner l'entreprenariat et ...
51,"EXG: ÉGALITÉ FEMMES-HOMMES, SEXUALITÉS ET GENRES",promouvoir l'égalité entre les femmes et les h...
64,FOOT LOISIRS ALPES EN MIXITE CHOISIE (FLAM),promouvoir la pratique féministe du football
151,FEMMES ENSEMBLE,valorisation et défense des droits des femmes
243,ENSEMBLE CONTRE LES VIOLENCES CONJUGALES,soutenir et aider les femmes victimes de viole...
278,AU CUR DE NOS ENFANTS,lutter contre les mutilations génitales féminines
348,UN TOIT SANS TOI,héberger des victimes de violences conjugales
419,"ELLE'P ECOUTER,LIBERER,LUTTER,ENCOURAGER,PROTEGER",aide aux victimes de violences intra-familiales
484,PHAROS-74 SOUTIEN AUX HOMMES VICTIMES DE VIOLE...,soutien aux hommes victimes de violences conju...
496,LES MAMANS,lutte pour la condition féminine


In [4]:
# Count identifiable beneficiaries in each year
funding_history = funding[
    funding["siren"].str.fullmatch(r"\d{9}", na=False)
].copy()

annual_coverage = funding_history.groupby("year").agg(
    record_count=("siren", "size"),
    identified_beneficiaries=("siren", "nunique")
)

display(annual_coverage)

# Count how many years each beneficiary appears in the dataset
years_per_beneficiary = funding_history.groupby("siren")["year"].nunique()

display(
    years_per_beneficiary.value_counts()
    .sort_index()
    .rename_axis("years_observed")
    .to_frame("beneficiary_count")
)

,record_count,identified_beneficiaries
year,,
2020,5254,2739
2021,6042,3339
2022,6432,3494
2023,6217,3368
2024,6362,3520


,beneficiary_count
years_observed,
1,2568
2,1152
3,633
4,581
5,1473


In [5]:
# Measure next-year presence among beneficiaries observed each year
continuity_results = []

for year in [2020, 2021, 2022, 2023]:
    current_beneficiaries = set(
        funding_history.loc[
            funding_history["year"] == year, "siren"
        ]
    )

    next_beneficiaries = set(
        funding_history.loc[
            funding_history["year"] == year + 1, "siren"
        ]
    )

    returning_count = len(
        current_beneficiaries.intersection(next_beneficiaries)
    )

    continuity_results.append({
        "starting_year": year,
        "beneficiary_count": len(current_beneficiaries),
        "observed_next_year": returning_count,
        "not_observed_next_year": (
            len(current_beneficiaries) - returning_count
        ),
        "return_share_pct": round(
            returning_count / len(current_beneficiaries) * 100, 1
        )
    })

continuity_summary = pd.DataFrame(continuity_results)
display(continuity_summary)


,starting_year,beneficiary_count,observed_next_year,not_observed_next_year,return_share_pct
0,2020,2739,2023,716,73.9
1,2021,3339,2372,967,71.0
2,2022,3494,2382,1112,68.2
3,2023,3368,2452,916,72.8


In [6]:
# Check continuity within program 137
program_137 = funding_history[
    funding_history["budget_program"] == "137"
]

program_results = []

for year in [2020, 2021, 2022, 2023]:
    current = set(
        program_137.loc[program_137["year"] == year, "siren"]
    )
    following = set(
        program_137.loc[program_137["year"] == year + 1, "siren"]
    )

    returning = len(current.intersection(following))

    program_results.append({
        "starting_year": year,
        "beneficiary_count": len(current),
        "observed_next_year_in_137": returning,
        "not_observed_next_year_in_137": len(current) - returning,
        "return_share_pct": (
            round(returning / len(current) * 100, 1)
            if current else None
        )
    })

display(pd.DataFrame(program_results))

,starting_year,beneficiary_count,observed_next_year_in_137,not_observed_next_year_in_137,return_share_pct
0,2020,55,46,9,83.6
1,2021,65,51,14,78.5
2,2022,63,54,9,85.7
3,2023,74,61,13,82.4


In [7]:
# Check other state programs for beneficiaries of program 137
complementary_results = []

for year in [2020, 2021, 2022, 2023, 2024]:
    annual_funding = funding_history[
        funding_history["year"] == year
    ]

    beneficiaries_137 = set(
        annual_funding.loc[
            annual_funding["budget_program"] == "137", "siren"
        ]
    )

    other_programs = annual_funding[
        annual_funding["budget_program"].notna()
        & (annual_funding["budget_program"] != "137")
    ]

    beneficiaries_other = set(other_programs["siren"])

    with_complementary = len(
        beneficiaries_137.intersection(beneficiaries_other)
    )

    complementary_results.append({
        "year": year,
        "beneficiaries_137": len(beneficiaries_137),
        "with_other_programs": with_complementary,
        "without_other_programs": (
            len(beneficiaries_137) - with_complementary
        ),
        "with_other_programs_pct": round(
            with_complementary / len(beneficiaries_137) * 100, 1
        ) if beneficiaries_137 else None
    })

display(pd.DataFrame(complementary_results))

,year,beneficiaries_137,with_other_programs,without_other_programs,with_other_programs_pct
0,2020,55,44,11,80.0
1,2021,65,48,17,73.8
2,2022,63,50,13,79.4
3,2023,74,59,15,79.7
4,2024,76,53,23,69.7


In [8]:
# Check next-year payments from programs other than 137
next_year_results = []

for year in [2020, 2021, 2022, 2023]:
    beneficiaries_137 = set(
        funding_history.loc[
            (funding_history["year"] == year)
            & (funding_history["budget_program"] == "137"),
            "siren"
        ]
    )

    next_year_other = set(
        funding_history.loc[
            (funding_history["year"] == year + 1)
            & funding_history["budget_program"].notna()
            & (funding_history["budget_program"] != "137"),
            "siren"
        ]
    )

    observed_count = len(
        beneficiaries_137.intersection(next_year_other)
    )

    next_year_results.append({
        "starting_year": year,
        "beneficiaries_137": len(beneficiaries_137),
        "other_programs_next_year": observed_count,
        "no_other_programs_observed_next_year": (
            len(beneficiaries_137) - observed_count
        ),
        "other_programs_next_year_pct": round(
            observed_count / len(beneficiaries_137) * 100, 1
        ) if beneficiaries_137 else None
    })

display(pd.DataFrame(next_year_results))

,starting_year,beneficiaries_137,other_programs_next_year,no_other_programs_observed_next_year,other_programs_next_year_pct
0,2020,55,46,9,83.6
1,2021,65,47,18,72.3
2,2022,63,48,15,76.2
3,2023,74,55,19,74.3


In [9]:
# Identify other programs not observed in the previous year
new_program_results = []

for year in [2020, 2021, 2022, 2023]:
    current = funding_history[funding_history["year"] == year]
    following = funding_history[funding_history["year"] == year + 1]

    beneficiaries_137 = set(
        current.loc[current["budget_program"] == "137", "siren"]
    )

    new_program_count = 0

    for siren in beneficiaries_137:
        current_programs = set(
            current.loc[
                current["siren"] == siren, "budget_program"
            ].dropna()
        )

        following_programs = set(
            following.loc[
                following["siren"] == siren, "budget_program"
            ].dropna()
        )

        # Exclude program 137 and programs already observed
        new_programs = following_programs - current_programs - {"137"}

        if new_programs:
            new_program_count += 1

    new_program_results.append({
        "starting_year": year,
        "beneficiaries_137": len(beneficiaries_137),
        "with_new_other_program_next_year": new_program_count,
        "share_pct": round(
            new_program_count / len(beneficiaries_137) * 100, 1
        ) if beneficiaries_137 else None
    })

display(pd.DataFrame(new_program_results))

,starting_year,beneficiaries_137,with_new_other_program_next_year,share_pct
0,2020,55,29,52.7
1,2021,65,26,40.0
2,2022,63,22,34.9
3,2023,74,23,31.1


In [12]:
# Compare access to additional programs with and without program 137
comparison_results = []

for year in [2020, 2021, 2022, 2023]:
    current = funding_history[
        funding_history["year"] == year
    ].dropna(subset=["budget_program"])

    following = funding_history[
        funding_history["year"] == year + 1
    ].dropna(subset=["budget_program"])

    # Collect the programs received by each beneficiary
    current_programs = current.groupby("siren")["budget_program"].agg(list)
    following_programs = following.groupby("siren")["budget_program"].agg(list)

    for siren, programs in current_programs.items():
        # Convert lists to sets to compare distinct programs
        programs = set(programs)
        next_programs = set(following_programs.get(siren, []))

        new_other_programs = next_programs - programs - {"137"}

        comparison_results.append({
            "starting_year": year,
            "siren": siren,
            "funded_by_137": "137" in programs,
            "new_other_program_next_year": bool(new_other_programs)
        })

program_access = pd.DataFrame(comparison_results)

# Summarize the results for each year and group
comparison = program_access.groupby(
    ["starting_year", "funded_by_137"]
).agg(
    beneficiary_count=("siren", "size"),
    new_program_count=("new_other_program_next_year", "sum"),
    new_program_share=("new_other_program_next_year", "mean")
)

comparison["new_program_share_pct"] = (
    comparison["new_program_share"] * 100
).round(1)

comparison = comparison.drop(columns=["new_program_share"])

display(comparison)

beneficiary_count  new_program_count  \
starting_year funded_by_137                                         
2020          False                       2684                775   
              True                          55                 29   
2021          False                       3274                684   
              True                          65                 26   
2022          False                       3431                570   
              True                          63                 22   
2023          False                       3294                626   
              True                          74                 23   

                             new_program_share_pct  
starting_year funded_by_137                         
2020          False                           28.9  
              True                            52.7  
2021          False                           20.9  
              True                            40.0  
2022          False                           16.6  
              True                            34.9  
2023          False                           19.0  
              True                            31.1

In [13]:
# Identify beneficiaries of program 137 in each year
beneficiaries_137 = funding_history.loc[
    funding_history["budget_program"] == "137",
    ["year", "siren"]
].drop_duplicates()

# Keep their payments from other programs in the same year
complementary_payments = funding_history[
    funding_history["budget_program"].notna()
    & (funding_history["budget_program"] != "137")
].merge(
    beneficiaries_137,
    on=["year", "siren"],
    how="inner",
    validate="many_to_one"
)

# Summarize complementary programs over 2020–2024
program_opportunities = complementary_payments.groupby(
    "budget_program"
).agg(
    beneficiary_count=("siren", "nunique"),
    years_observed=("year", "nunique"),
    total_amount=("amount", "sum")
)

# Identify programs also received by CIDFF during the period
cidff_programs = set(
    funding_history.loc[
        funding_history["siren"] == "305933517",
        "budget_program"
    ].dropna()
)

program_opportunities["observed_for_cidff"] = (
    program_opportunities.index.isin(cidff_programs)
)

display(
    program_opportunities.sort_values(
        "beneficiary_count", ascending=False
    ).head(15)
)

,beneficiary_count,years_observed,total_amount,observed_for_cidff
budget_program,,,,
163,40,5,1.584213e+06,True
147,28,5,4.779388e+06,True
216,25,5,3.302633e+06,True
304,21,5,4.541651e+06,True
129,18,5,8.588181e+05,False
177,16,5,1.284784e+08,False
107,16,5,2.423973e+06,False
101,11,5,9.997546e+06,False
182,11,5,8.753133e+05,False


In [14]:
# Inspect programs not observed for CIDFF during 2020–2024
programs_to_review = (
    program_opportunities.loc[
        ~program_opportunities["observed_for_cidff"]
    ]
    .sort_values("beneficiary_count", ascending=False)
    .head(3)
    .index
)

for program in programs_to_review:
    print("\nBudget program:", program)

    examples = complementary_payments.loc[
        complementary_payments["budget_program"] == program,
        ["year", "association_name", "funding_purpose"]
    ].sort_values("year", ascending=False)

    # Show one example per recorded beneficiary name
    display(
        examples.drop_duplicates("association_name").head(5)
    )


Budget program: 129


,year,association_name,funding_purpose
900,2024,ASS POUR REHABILITATION DES,Subvention dotation aux territoires MILDECA
899,2024,ASSOCIATION LE MAS,Subvention dotation aux territoires MILDECA
898,2024,ISSUE DE SECOURS : MAISON-RELAIS ET,Subvention dotation aux territoires MILDECA
896,2024,LE LIEN THEATRE,DILCRAH - Appel à projet déconcentré
904,2024,FONDATION GEORGES BOISSEL,Subvention MILDECA FDC- dotation minju



Budget program: 177


,year,association_name,funding_purpose
1034,2024,VIFFIL - SOS FEMMES,Hébergement d'urgence
1037,2024,ASS POUR REHABILITATION DES,Hébergement social
1038,2024,ESPACE FEMMES GENEVIEVE D.,Hébergement d'urgence
1032,2024,AU TAMBOUR !,Action sociale sans hébergement
1031,2024,ASSOCIATION LE MAS,Hébergement d'urgence



Budget program: 107


,year,association_name,funding_purpose
884,2024,ENDOMIND FRANCE,NaN
892,2024,FONDATION GEORGES BOISSEL,NaN
893,2024,ASSOCIATION LE MAS,NaN
895,2024,LA SASSON,NaN
891,2024,COMITE DAUPHINOIS ACTION SOCIO,NaN


In [15]:
from pathlib import Path

# Find possible HCE survey files
for file in Path(".").rglob("*.csv"):
    if any(word in file.name.lower() for word in ["hce", "toluna", "sexisme"]):
        print(file)

base-de-donnees-toluna-harris-barometre-sexisme-vague-5-2025-dgcs-hce-.csv


In [16]:
# Load the HCE survey and inspect the available variables
hce = pd.read_csv(
    "base-de-donnees-toluna-harris-barometre-sexisme-vague-5-2025-dgcs-hce-.csv",
    sep=";",
    dtype="string"
)

print("Rows and columns:", hce.shape)
print(hce.columns.tolist())

Rows and columns: (3061, 264)
['responseid', 'SEXE : SEXE - Vous êtes :', 'AGE_1 : AGE_1 (AGE - Veuillez nous indiquer votre âge :)', 'REC_uda5 : REC_uda5 - Recodage en région UDA5 (question cachée)', 'REC_REGION_NEW : REC_REGION_NEW - Recodage nouvelles\xa0régions :', 'REC_AGGLO_QUOTA : REC_AGGLO_QUOTA - Taille d’agglomération pour quotas', 'REC_TailleAgglo5 : REC_TailleAgglo5', 'REC_AGE3 : REC_AGE3', 'ACTIVITE_REP : ACTIVITE_REP - Actuellement, quelle est votre situation principale ?', 'PCS_REP : PCS_REP - Dans quelle catégorie socioprofessionnelle vous situez-vous ?', 'RECOD_PCS_REPONDANT : RECOD_PCS_REPONDANT - RECOD_PCS_REPONDANT -PCS Répondant', 'RECOD_PCS_REPONDANT2 : RECOD_PCS_REPONDANT2 - RECOD_PCS_REPONDANT2 -PCS Répondant', 'FRERES : FRERES - Avez-vous un / des frères ?', 'SOEURS : SOEURS - Avez-vous une / des sœurs ?', 'ENCEINTE : ENCEINTE - Avez-vous déjà été enceinte ?', 'ENFANTS : ENFANTS - Avez-vous des enfants ?', 'SEXE_ENFANTS_1 : Un/des garçons (SEXE_ENFANTS - Avez-v

In [17]:
# Inspect two possible targets before choosing a research question
for code in ["Q18", "Q25_1"]:
    column = next(
        name for name in hce.columns
        if name.startswith(code + " :")
    )

    print("\n" + column)
    display(
        hce[column].value_counts(dropna=False)
        .rename_axis("response")
        .to_frame("respondent_count")
    )


Q18 : Q18 - Quel niveau d’information pensez-vous avoir sur les lois et sanctions existantes pour lutter contre le sexisme ? Vous vous sentez… ?


,respondent_count
response,
Plutôt informé(e),1247
Plutôt pas informé(e),1180
Pas du tout informé(e),296
Très informé(e),228
Non réponse / Ne sait pas,110



Q25_1 : Les associations spécialisées (Q25 - Faites-vous confiance à ces institutions pour prévenir et lutter contre les actes et violences sexistes et sexuelles ?)


,respondent_count
response,
Oui plutôt,1509
Oui tout à fait,802
Non plutôt pas,372
Non pas du tout,210
Non réponse / Ne sait pas,168


In [18]:
# Inspect responses to the proposed target question
target_column = next(
    column for column in hce.columns
    if column.startswith("Q4_19 :")
)

print(target_column)

display(
    hce[target_column]
    .value_counts(dropna=False)
    .rename_axis("response")
    .to_frame("respondent_count")
)

Q4_19 : Le féminisme menace la place et le rôle des hommes dans la société (Q4 - Dans quelle mesure êtes-vous d’accord avec les affirmations suivantes ?)


,respondent_count
response,
Pas du tout d’accord,948
Plutôt pas d’accord,858
Plutôt d’accord,646
Tout à fait d’accord,327
Non réponse / Ne sait pas,282


In [19]:
# Inspect possible sociodemographic predictors
predictor_codes = [
    "SEXE", "AGE_1", "DIPLOME",
    "ACTIVITE_REP", "PCS_REP",
    "REC_REGION_NEW", "REC_TailleAgglo5"
]

for code in predictor_codes:
    matches = [
        column for column in hce.columns
        if column.startswith(code + " :")
    ]

    if matches:
        column = matches[0]
        print("\n" + column)

        display(
            hce[column].value_counts(dropna=False)
            .rename_axis("value")
            .to_frame("respondent_count")
        )


SEXE : SEXE - Vous êtes :


,respondent_count
value,
Une femme,1610
Un homme,1451



AGE_1 : AGE_1 (AGE - Veuillez nous indiquer votre âge :)


,respondent_count
value,
18,91
66,84
65,76
45,71
25,70
...,...
86,2
81,2
84,1



DIPLOME : DIPLOME - Quel est le diplôme le plus élevé que vous ayez obtenu ?


,respondent_count
value,
Baccalauréat d’enseignement technique ou général,624
"Bac+2 (BTS, DUT, DEUST, Diplôme universitaire de niveau Bac+2 comme l’ancien DEUG)",534
"Bac+3 (Licence, Bachelor, Licence professionnelle)",370
Baccalauréat d’enseignement professionnel,271
"Bac+5 (Master 2, diplôme d’ingénieur ou équivalent, DEA, DESS)",263
Certificat d’aptitude professionnelle (CAP),254
Brevet d’études professionnelles (BEP),176
Brevet des collèges / BEPC / Brevet (ou équivalent),161
"Bac+4 (Master 1, ancienne Maîtrise ou équivalent)",148


... (4 outputs left out)


REC_REGION_NEW : REC_REGION_NEW - Recodage nouvelles régions :


,respondent_count
value,
Île de France,571
Auvergne-Rhône-Alpes,382
Hauts-de-France,309
Occitanie,275
Grand Est,265
Nouvelle-Aquitaine,259
Provence-Alpes-Côte d’Azur,225
Pays de la Loire,200
Bretagne,162



REC_TailleAgglo5 : REC_TailleAgglo5


,respondent_count
value,
Unité urbaine de 100 000 à 1 999 999 habitants,1071
Unité urbaine de Paris,515
Zone rurale,505
Unité urbaine de 20 000 à 99 999 habitants,493
Unité urbaine de 2 000 à 19 999 habitants,477


In [20]:
# Check respondent identifiers and survey weights
print("Missing respondent IDs:", hce["responseid"].isna().sum())
print("Duplicate respondent IDs:", hce["responseid"].duplicated().sum())

# Convert decimal commas if needed
survey_weights = pd.to_numeric(
    hce["POIDS"].str.replace(",", ".", regex=False),
    errors="coerce"
)

print("\nMissing or invalid weights:", survey_weights.isna().sum())
print("Zero or negative weights:", survey_weights.le(0).sum())

display(survey_weights.describe().round(3))

Missing respondent IDs: 0
Duplicate respondent IDs: 0

Missing or invalid weights: 0
Zero or negative weights: 0


count    3061.0
mean        1.0
std       0.147
min       0.785
25%       0.898
50%       0.963
75%       1.022
max       1.608
Name: POIDS, dtype: Float64

## Conclusion — Selection of the ML question

This feasibility study explored association missions, funding histories
and HCE survey responses.

Classifying association missions would require manually validated labels.
Funding prediction was also considered. However, the territorial program
137 sample was small, and future payments may be affected by changes in
public policy. Its practical value for CIDFF remained uncertain.

The selected question is:

**Can sociodemographic characteristics predict agreement with the statement
“Feminism threatens men's place and role in society”?**

The HCE dataset contains 2,779 expressed responses to this question:
973 respondents agree and 1,806 disagree. The 282 non-response or
“Don't know” answers are excluded from the binary target.

Five initial predictors were selected: declared sex, age, education,
employment status and urban area size. They have no missing values among
the retained respondents. Respondent identifiers are unique, and survey
weights are valid and positive.

This supervised classification task connects with CIDFF's reflection on
antifeminist representations and prevention themes. It also allows us to
apply the methods studied during the course: a simple baseline, logistic
regression and Random Forest, followed by validation and justified tuning.

The analysis predicts one declared attitude. It does not identify
masculinist individuals, dangerous behaviour or causes of violence.
National survey results do not directly describe CIDFF users or establish
which awareness activities would be effective.

Model development and evaluation are presented separately in
`06_hce_classification.ipynb`. No predictive performance is established
at this feasibility stage.